In [1]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# fn_ERA5 = f'/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/ERA5_{year}.zarr'
# ds_ERA5 = xr.open_zarr(fn_ERA5)
# fn_CESM = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year-1}-11-01_daily_ensemble.zarr'
# ds_CESM = xr.open_zarr(fn_CESM)

### Preparing gridded yearly metrics

In [4]:
dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}
keys = list(dict_loc.keys())

### Fairbanks: TMAX

In [5]:
key = 'Fairbanks'
dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{key}/'
base_dir = '/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/'

In [8]:
list_ds = []

for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'/{key}/ERA5_{key}_{year}.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)[[
        '2m_temperature', 
        'maximum_2m_temperature_since_previous_post_processing'
    ]]
    
    ds_ERA5 = ds_ERA5.rename(
        {
            '2m_temperature': "TREFHT",
            'maximum_2m_temperature_since_previous_post_processing': "TREFHTMX",
        }
    )


    ds_group_MX = ds_ERA5[["TREFHTMX"]].groupby("time.year")
    ds_group_MEAN = ds_ERA5[["TREFHT"]].groupby("time.year")
    
    ds_max_hourly = ds_group_MX.max(dim="time",  skipna=True)
    ds_max_daily = ds_group_MEAN.max(dim="time",  skipna=True)
    ds_max_7d = ds_group_MEAN.map(
        lambda x: x.rolling(time=7, min_periods=7).mean().max(dim="time", skipna=True)
    )
    ds_max_30d = ds_group_MEAN.map(
        lambda x: x.rolling(time=30, min_periods=30).mean().max(dim="time", skipna=True)
    )

    ds_max_hourly = ds_max_hourly.rename({"TREFHTMX": "TREFHTMX_max"})
    ds_max_daily = ds_max_daily.rename({v: f"{v}_max" for v in ds_max_daily.data_vars})
    ds_max_7d = ds_max_7d.rename({v: f"{v}_7d_max"  for v in ds_max_7d.data_vars})
    ds_max_30d = ds_max_30d.rename({v: f"{v}_30d_max"  for v in ds_max_30d.data_vars})

    # ============ #
    ds_merge = xr.merge([ds_max_hourly, ds_max_daily, ds_max_7d, ds_max_30d])
    list_ds.append(ds_merge)

ds_all = xr.concat(list_ds, dim='year')
ds_all = ds_all.chunk({'year': 62, 'lat': 21, 'lon': 16})

In [11]:
ds_all

<xarray.Dataset>
Dimensions:         (year: 68, lat: 21, lon: 16)
Coordinates:
  * lat             (lat) float64 55.13 56.07 57.02 57.96 ... 72.09 73.04 73.98
  * lon             (lon) float64 203.8 205.0 206.2 207.5 ... 220.0 221.2 222.5
  * year            (year) int64 1958 1959 1960 1961 ... 2022 2023 2024 2025
Data variables:
    TREFHTMX_max    (year, lat, lon) float32 dask.array<chunksize=(62, 21, 16), meta=np.ndarray>
    TREFHT_max      (year, lat, lon) float32 dask.array<chunksize=(62, 21, 16), meta=np.ndarray>
    TREFHT_7d_max   (year, lat, lon) float32 dask.array<chunksize=(62, 21, 16), meta=np.ndarray>
    TREFHT_30d_max  (year, lat, lon) float32 dask.array<chunksize=(62, 21, 16), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

In [10]:
save_name = dir_stn + 'ERA5_T2_max.zarr'
# ds_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/ERA5_T2_max.zarr
